In [2]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin

# Define the Zero-Rule Baseline Classifier directly
class ZeroRuleClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.majority_class_ = self.classes_[np.argmax(counts)]
        return self
    def predict(self, X):
        return np.array([self.majority_class_] * len(X))
    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

print("Libraries and ZeroRuleClassifier loaded successfully!")

Libraries and ZeroRuleClassifier loaded successfully!


In [ ]:
# Load from the correct folder path 'data_preparation/processed/'
train_path = "../data_preparation/processed/train_cleaned.csv"
val_path = "../data_preparation/processed/validation_cleaned.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

print("Cleaned Training Dataset Shape:", train_df.shape)
print("Cleaned Validation Dataset Shape:", val_df.shape)

Cleaned Training Dataset Shape: (19300, 530)
Cleaned Validation Dataset Shape: (1111, 530)


In [ ]:
wap_cols = [col for col in train_df.columns if col.startswith('WAP')]

X_train_raw = train_df[wap_cols]
y_train_raw = train_df["LOCATION_CLASS"]

X_holdout = val_df[wap_cols]
y_holdout = val_df["LOCATION_CLASS"]

# 1. Perform stratified split FIRST (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_train_raw
)

# 2. Fit StandardScaler ONLY on the training set, then transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=wap_cols)
X_test = pd.DataFrame(X_test_scaled, columns=wap_cols)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

Training feature shape: (15440, 520)
Testing feature shape: (3860, 520)


In [9]:
from sklearn.metrics import accuracy_score, f1_score

# Evaluate Zero-Rule Baseline
baseline = ZeroRuleClassifier()
baseline.fit(X_train, y_train)

preds = baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, preds)
baseline_f1 = f1_score(y_test, preds, average='macro')

print(f"Zero-Rule Baseline Test Accuracy: {baseline_acc * 100:.2f}%")
print(f"Zero-Rule Baseline Macro-F1: {baseline_f1:.4f}")

Zero-Rule Baseline Test Accuracy: 14.04%
Zero-Rule Baseline Macro-F1: 0.0189


**Analysis** 
We have established the minimum performance level that we can use to compare the rest of our machine leanring models to make sure they are learning something useful. An accuracy of 14.04% means that it correctly predicted that many locations and it indicates how frequently the most common location occurs in the dataset. The macro-f1 score was incredibly low because the zero rule only predicts one location.
